# SoccerNet trajectory experiments (local machine)

Local copy of **`colab_atharv.ipynb`**: same flow without Google Drive or `/content`.

**Prerequisites:** activate the project venv (e.g. `source .venv/bin/activate`) and open this notebook so the kernel uses that Python.

Colab version: [`colab_atharv.ipynb`](colab_atharv.ipynb).


In [ ]:
from pathlib import Path

def find_repo() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *here.parents[:6]]
    for p in candidates:
        if (p / 'pyproject.toml').is_file() and (p / 'src' / 'ballcond').is_dir():
            return p
    raise RuntimeError(
        'Could not find repo root (pyproject.toml + src/ballcond). '
        'Set REPO_DIR manually to your cv-project path.'
    )

REPO_DIR = find_repo()
DATA_DIR = Path.home() / 'data' / 'soccernet'
TRACKING_ROOT = DATA_DIR / 'tracking'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('REPO_DIR      ', REPO_DIR)
print('DATA_DIR      ', DATA_DIR)
print('TRACKING_ROOT ', TRACKING_ROOT)


In [ ]:
import os
os.chdir(REPO_DIR)
!pip install -q SoccerNet
!pip install -q -r requirements.txt
!pip install -q -e .

import sys
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))


## Download SoccerNet-Tracking

The **`SoccerNetDownloader`** WebDAV URLs often return **401** on current servers. This repo uses **`soccernet_detect.nc_tracking_download`** (Nextcloud share login), then unpacks zips the same way as `scripts/setup_soccernet_yolo_local.py`.

Share password is usually **`SoccerNet`** (not `s0cc3rn3t`). Large download.


In [ ]:
import importlib.util
from pathlib import Path
from soccernet_detect.nc_tracking_download import download_tracking_zips

download_tracking_zips(Path(DATA_DIR), ['train', 'test', 'challenge'], 'SoccerNet')

_spec = importlib.util.spec_from_file_location('_sl', REPO_DIR / 'scripts' / 'setup_soccernet_yolo_local.py')
_sl = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_sl)
_sl.extract_tracking_archives(Path(TRACKING_ROOT), splits=['train', 'test', 'challenge'], force=False)
print('Done. Expect', TRACKING_ROOT / 'train' / 'SNMOT-*')


In [ ]:
# Optional: legacy pip downloader (often 401 locally). Uncomment only if you know your host works.
# from SoccerNet.Downloader import SoccerNetDownloader
# dl = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
# # dl.downloadDataTask(task='tracking', split=['train', 'test', 'challenge'])
# # dl.downloadDataTask(task='tracking', split=['train', 'test', 'challenge'], password='SoccerNet')


## Explore SoccerNet-Tracking layout

Official clips include **`gameinfo.ini`** next to `gt/` and `seqinfo.ini`. The training loader uses it to:

- pick the **ball** track id (`trackletID_<id>= ball;…`) instead of bbox-area heuristics;
- keep only **field players + goalkeepers** as prediction targets (referees/staff dropped);
- attach **team side** (left vs right in frame) for learned team embeddings.

`gt/gt.txt` is still MOT format (columns 8–10 are `-1`); class names live in **gameinfo**, not in the CSV.


In [ ]:
import os
for split in ['train', 'test', 'challenge']:
    split_dir = TRACKING_ROOT / split
    if split_dir.is_dir():
        seqs = sorted(os.listdir(split_dir))
        print(f'{split}: {len(seqs)} sequences — {seqs[:5]}{'...' if len(seqs) > 5 else ''}')
    else:
        print(f'{split}: not found')


In [ ]:
import pandas as pd
from pathlib import Path

sample_dir = TRACKING_ROOT / 'train' / 'SNMOT-060'
sample_gt = sample_dir / 'gt' / 'gt.txt'
sample_gi = sample_dir / 'gameinfo.ini'

df = pd.read_csv(sample_gt, header=None)
df.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'c1', 'c2', 'c3'][: df.shape[1]]
print(f'gt.txt shape: {df.shape}')
print(df.head(5).to_string(index=False))

print(f'\n--- {sample_gi.name} (tracklet lines) ---')
if sample_gi.is_file():
    for line in sample_gi.read_text().splitlines():
        if line.lower().startswith('trackletid_'):
            print(line)
else:
    print('gameinfo.ini missing — finish download/extract.')


## Loader sanity check (`ballcond.data.soccernet`)

Uses **`gameinfo.ini`** for ball id, **field players + GK** only, and **team** labels (`player_team`: 0=left, 1=right, 2=unknown).


In [ ]:
from pathlib import Path

cfg = REPO_DIR / 'configs'
paths = sorted(cfg.glob('soccernet_exp_*.yaml'))
bc = cfg / 'soccernet_ballcond.yaml'
if bc.is_file():
    paths.append(bc)
for p in paths:
    print(' ', p.relative_to(REPO_DIR))


In [ ]:
from pathlib import Path
import numpy as np
from ballcond.data.soccernet import load_soccernet_sequence, load_soccernet_split

seq = load_soccernet_sequence(TRACKING_ROOT / 'train' / 'SNMOT-060')
print(f'Sequence: {seq.sequence_id}')
print(f'Frames: {seq.num_frames}, Field players + GK columns: {seq.num_players}')
print(f'Ball track id: {seq.meta.get("ball_track_id")} (source: {seq.meta.get("ball_track_source")})')
if seq.has_ball:
    print(f'Ball observed in {seq.ball_mask.sum()}/{seq.num_frames} frames')
if seq.player_team is not None:
    u, c = np.unique(seq.player_team, return_counts=True)
    labels = {0: 'team_left', 1: 'team_right', 2: 'unknown'}
    print('Team label counts:', {labels[int(t)]: int(n) for t, n in zip(u, c)})
print(f'Scale (W, H): {seq.scale}')


In [ ]:
train_seqs = load_soccernet_split(TRACKING_ROOT / 'train')
print(f'Train sequences: {len(train_seqs)}')

n_with_ball = sum(1 for s in train_seqs if s.has_ball)
src_gi = sum(1 for s in train_seqs if s.meta.get('ball_track_source') == 'gameinfo')
print(f'With ball tensor: {n_with_ball}/{len(train_seqs)}  |  ball id from gameinfo: {src_gi}/{len(train_seqs)}')

for s in train_seqs[:5]:
    src = s.meta.get('ball_track_source', '?')
    bid = s.meta.get('ball_track_id')
    vis = f'{s.ball_mask.sum()}/{s.num_frames}' if s.has_ball else 'n/a'
    print(f'  {s.sequence_id}: {s.num_frames}f, {s.num_players} targets, ball={bid} ({src}), visible={vis}')


In [ ]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('CUDA not available — training will use CPU.')


## Training runs

Configs ship with Colab `data.root` paths. **This notebook overrides `data.root` to `TRACKING_ROOT`** when you run the cells below (temporary YAML in `/tmp`).

Each run saves to `results/<run_name>/`.

### Registry: all `model.name` values

| Kind | `model.name` | Notes |
|------|--------------|--------|
| Baseline | `kalman` | Constant-velocity; use `model.name: kalman` (no `kwargs`). |
| Baseline | `lstm` | Per-player LSTM; ignores ball and other agents. |
| Trajectory | `transformer_symmetric` | Factored **time × agent** attention; optional `use_ball`. |
| Trajectory | `transformer_ball_broadcast` | Ball pooled → vector added to all player **time** tokens. |
| Trajectory | `transformer_ballcond` | Players **cross-attend** to encoded ball timeline. |
| Entity | `transformer_entity` | **One token per player** (flatten last `entity_history` steps → linear → self-attn). |
| Entity | `transformer_entity_ball_symmetric` | + **one token per ball track** in the same self-attn set. |
| Entity | `transformer_entity_ball_joint` | + **mean ball embedding** broadcast-added to each player, then self-attn over players only. |

### SoccerNet — time × agent (A–C) + optional BallCond

| Exp | Config | Model |
|-----|--------|-------|
| **A** | `soccernet_exp_ball_symmetric.yaml` | `transformer_symmetric` |
| **B** | `soccernet_exp_ball_broadcast.yaml` | `transformer_ball_broadcast` |
| **C** | `soccernet_exp_no_ball.yaml` | `transformer_symmetric` (`use_ball: false`) |
| Optional | `soccernet_ballcond.yaml` | `transformer_ballcond` |

### SoccerNet — entity-level (D–F)

Match `entity_history` to `data.history` (20 below).

| Exp | Config | Model |
|-----|--------|-------|
| **D** | `soccernet_exp_entity.yaml` | `transformer_entity` |
| **E** | `soccernet_exp_entity_ball_symmetric.yaml` | `transformer_entity_ball_symmetric` |
| **F** | `soccernet_exp_entity_ball_joint.yaml` | `transformer_entity_ball_joint` |

### A — Time × agent, ball as extra agents



In [ ]:
import os, subprocess, sys, tempfile
from omegaconf import OmegaConf

def train_local(config_name: str) -> None:
    cfg_path = REPO_DIR / 'configs' / config_name
    cfg = OmegaConf.load(cfg_path)
    if cfg.data.get('name') == 'soccernet':
        cfg.data.root = str(TRACKING_ROOT.resolve())
    fd, tmp = tempfile.mkstemp(suffix='.yaml')
    os.close(fd)
    OmegaConf.save(cfg, tmp)
    subprocess.run([sys.executable, '-m', 'ballcond.train', '--config', tmp, '--out', 'results'], cwd=REPO_DIR, check=True)
    os.unlink(tmp)

train_local('soccernet_exp_ball_symmetric.yaml')


### B — Ball broadcast embedding


In [ ]:
train_local('soccernet_exp_ball_broadcast.yaml')


### C — No ball input


In [ ]:
train_local('soccernet_exp_no_ball.yaml')


### Optional — BallCond


In [ ]:
train_local('soccernet_ballcond.yaml')


### D — Entity: players only (`transformer_entity`)


In [ ]:
train_local('soccernet_exp_entity.yaml')


### E — Entity: one token per ball track (`transformer_entity_ball_symmetric`)


In [ ]:
train_local('soccernet_exp_entity_ball_symmetric.yaml')


### F — Entity: mean ball vector added to every player token (`transformer_entity_ball_joint`)


In [ ]:
train_local('soccernet_exp_entity_ball_joint.yaml')


## Compare metrics

Loads `metrics.json` from each `results/<run_name>/` after training.


In [ ]:
import json
import os

runs = [
    ('exp_A_symmetric', 'soccernet_ball_symmetric_agent'),
    ('exp_B_broadcast', 'soccernet_ball_broadcast'),
    ('exp_C_no_ball', 'soccernet_ball_none'),
    ('optional_ballcond', 'soccernet_ballcond'),
    ('exp_D_entity', 'soccernet_entity_players'),
    ('exp_E_entity_ball_sym', 'soccernet_entity_ball_symmetric'),
    ('exp_F_entity_ball_joint', 'soccernet_entity_ball_joint'),
]
results = {}
for label, dirname in runs:
    path_m = REPO_DIR / 'results' / dirname / 'metrics.json'
    if path_m.is_file():
        with open(path_m) as f:
            results[label] = json.load(f)['val']

if not results:
    print('No metrics yet — run training cells above.')
else:
    keys = sorted(next(iter(results.values())).keys())
    labels = list(results.keys())
    hdr = '{:12}'.format('Metric') + ''.join('{:>16}'.format(lb) for lb in labels)
    print(hdr)
    print('-' * len(hdr))
    for k in keys:
        parts = []
        for lb in labels:
            v = results[lb].get(k, float('nan'))
            parts.append('{:16.5f}'.format(v))
        print('{:12}'.format(k) + ''.join(parts))
